# RetrieveChat based FinRobot-RAG

In this demo, we showcase the RAG usecase of our finrobot, which inherits from autogen's RetrieveChat implementation.


Instead of using `RetrieveUserProxyAgent` directly, we register the context retrieval as a function for our bots.
For detailed implementation, refer to [rag function](../finrobot/functional/rag.py) and [rag workflow](../finrobot/agents/workflow.py) of `SingleAssistantRAG` 

In [1]:
import autogen
from finrobot.agents.workflow import SingleAssistantRAG

for openai configuration, rename OAI_CONFIG_LIST_sample to OAI_CONFIG_LIST and replace the api keys

In [2]:

llm_config = {
    "config_list": [{
        "model": "qwen-turbo",
        "api_key": "sk-33e77da7a560490cbc6e053c91e2e6a0",
        "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1",
        "max_tokens": 1000,
        "price": [0.0, 0.0]           
    }],
    "timeout": 600
}


From `finrobot.agents.workflow` we import the `SingleAssistantRAG`, which takes a `retrieve_config` as input.
For `docs_path`, we first put our generated pdf report from [this notebook](./agent_annual_report.ipynb). 

For more configuration, refer to [autogen's documentation](https://microsoft.github.io/autogen/docs/reference/agentchat/contrib/retrieve_user_proxy_agent)

Then, lets do a simple Q&A.

In [5]:
assitant = SingleAssistantRAG(
    "Data_Analyst",
    llm_config,
    human_input_mode="NEVER",
    retrieve_config={
        "task": "qa",
        "vector_db": None,  # Autogen has bug for this version
        "docs_path": [
            #"../report/Microsoft_Annual_Report_2023.pdf",
            "../report/nvidia_annual_report.pdf",
        ],
        "chunk_token_size": 1000,
        "get_or_create": True,
        "collection_name": "msft_analysis",
        "must_break_at_empty_line": False,
    },
)
assitant.chat("How's NVDA's 2025 income? Provide with some analysis,then translate in chinese.")

User_Proxy (to Data_Analyst):

How's NVDA's 2025 income? Provide with some analysis,then translate in chinese.

--------------------------------------------------------------------------------
Data_Analyst (to User_Proxy):


***** Suggested tool call (call_1ba84910560747f89a7af0): retrieve_content *****
Arguments: 
{"message": "NVDA's 2025 income", "n_results": 3}
*******************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION retrieve_content...
Trying to create collection.
doc_ids:  [['doc_0']]
Adding content of doc doc_0 to context.
User_Proxy (to Data_Analyst):

***** Response from calling tool (call_1ba84910560747f89a7af0) *****
Below is the context retrieved from the required file based on your query.
If you can't answer the question with or without the current context, you should try using a more refined search query according to your requirements, or as

Here we come up with a more complex case, where we put the 10-k report of MSFT here.

Let' see how the agent work this out.

In [7]:
assitant = SingleAssistantRAG(
    "Data_Analyst",
    llm_config,
    human_input_mode="NEVER",
    retrieve_config={
        "task": "qa",
        "vector_db": None,  # Autogen has bug for this version
        "docs_path": [#nvidia_annual_report.pdf
            #"../report/2023-07-27_10-K_msft-20230630.htm.pdf",
            "../report/nvidia_annual_report.pdf",
        ],
        "chunk_token_size": 2000,
        "collection_name": "nvda_10k",
        "get_or_create": True,
        "must_break_at_empty_line": False,
    },
    rag_description="Retrieve content from NVDA's 2025 10-K report for detailed question answering.",
)
assitant.chat("How's nvda's 2025 income?Give short reason, Provide with some analysis,then translate in chinese.")

User_Proxy (to Data_Analyst):

How's nvda's 2025 income?Give short reason, Provide with some analysis,then translate in chinese.

--------------------------------------------------------------------------------
Data_Analyst (to User_Proxy):


***** Suggested tool call (call_85d165caee5748809d9096): retrieve_content *****
Arguments: 
{"message": "2025 income of NVIDIA", "n_results": 3}
*******************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION retrieve_content...
Trying to create collection.
doc_ids:  [['doc_0']]
Adding content of doc doc_0 to context.
User_Proxy (to Data_Analyst):

***** Response from calling tool (call_85d165caee5748809d9096) *****
Below is the context retrieved from the required file based on your query.
If you can't answer the question with or without the current context, you should try using a more refined search query according to you